<center><h1 style="font-size: 30px;"><b>Athens House Listings for Sale Dataset Cleaning</b></h1></center>

<center><h1 style="font-size: 18px;"><b>This notebook serves as the dedicated data pre-processing pipeline for raw real estate listings.</b></h1></center>

As of April 2026, Spitogatos website has 37103 total listings for sale in the center of Athens. We managed to collect a random sample of 14501 listings for the analysis.  
In this notebook, we will only focus on cleaning the orgininal raw, web scraped dataset (irrelevant columns, duplicate and missing values).

In [1]:
# Import the necessary libraries
import pandas as pd
import warnings
import numpy as np

In [2]:
# Ignore all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
# Set path to retrieve csv file
file_path = r"C:\Users\marko\Documents\Database\Athens_House_Listings\Scraped\house_listings_dataset.csv"

In [4]:
# Load the csv file into the df
try:
    df = pd.read_csv(file_path)
    print("Csv file successfully loaded to the dataframe!")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Csv file successfully loaded to the dataframe!


In [5]:
# View the first rows
df.head()

,listingId,airConditioning,alarm,areaSqm,balcony,bathrooms,closedParkingSpots,currency,elevator,energyClass,...,openParkingSpots,pool,price,pricePerSqm,renovationYear,rooms,solarHeater,storage,view,yearBuilt
0,19435790,True,True,51,True,NaN,NaN,EUR,True,D,...,NaN,False,106500,2088.24,2020.0,1.0,False,False,True,1958.0
1,19457736,True,False,53,True,1.0,NaN,EUR,True,C,...,NaN,False,193000,3641.51,NaN,1.0,True,False,True,2003.0
2,19618641,False,True,112,True,1.0,NaN,EUR,True,C,...,NaN,False,295000,2633.93,2023.0,3.0,False,False,True,1980.0
3,18294034,True,True,135,NaN,2.0,NaN,EUR,True,B,...,NaN,False,1300000,9629.63,2025.0,2.0,False,False,NaN,NaN
4,18246939,True,True,304,True,2.0,NaN,EUR,True,D,...,NaN,False,3500000,11513.16,2025.0,3.0,NaN,True,NaN,1937.0


In [6]:
print('The dimensions of the dataset is:', df.shape)

The dimensions of the dataset is: (14501, 31)


In [7]:
print('Number of duplicates in the dataset:', df.duplicated().sum())

Number of duplicates in the dataset: 2285


In [8]:
df = df.drop_duplicates()
print('Number of duplicates after duplicates removal:', df.duplicated().sum())

Number of duplicates after duplicates removal: 0


In [9]:
print('Now the shape of the df is:', df.shape)

Now the shape of the df is: (12216, 31)


In [10]:
# View the data types of each column
df.dtypes

listingId               int64
airConditioning        object
alarm                  object
areaSqm                 int64
balcony                object
bathrooms             float64
closedParkingSpots    float64
currency               object
elevator               object
energyClass            object
fireplace              object
floor                   int64
furnished              object
garden                 object
hasParking             object
isNewDevelopment         bool
kitchens              float64
latitude              float64
livingRooms           float64
location               object
longitude             float64
openParkingSpots      float64
pool                   object
price                   int64
pricePerSqm           float64
renovationYear        float64
rooms                 float64
solarHeater            object
storage                object
view                   object
yearBuilt             float64
dtype: object

In [11]:
# View missing values for each column
df.isnull().sum()

listingId                 0
airConditioning         199
alarm                   308
areaSqm                   0
balcony                 137
bathrooms               364
closedParkingSpots    11729
currency                  0
elevator                123
energyClass              11
fireplace               301
floor                     0
furnished               286
garden                  338
hasParking              300
isNewDevelopment          0
kitchens               1416
latitude                  0
livingRooms            3138
location                  0
longitude                 0
openParkingSpots      11494
pool                    498
price                     0
pricePerSqm               0
renovationYear         7287
rooms                   273
solarHeater             553
storage                 269
view                    710
yearBuilt               167
dtype: int64

# Columns Inspection

This section outlines strategies for addressing missing values to maintain data integrity and ensure the accuracy of our analysis.

### "listingId":

No missing values, but we **don't need this column** for the analysis so it **must be dropped**.

In [12]:
df = df.drop(columns=['listingId'])

### "airConditioning":

**1.6%** of the total values is missing. These values will be filled with the **most common category**.

In [13]:
df['airConditioning'].mode()

0    True
Name: airConditioning, dtype: object

In [14]:
df['airConditioning'] = df['airConditioning'].fillna(True).astype(bool)

In [15]:
print('Missing values in "airConditioning" column after filling:', df['airConditioning'].isnull().sum())

Missing values in "airConditioning" column after filling: 0


### "alarm":

**2.5%** of the total values is missing. These values will be filled with the **most common category**.

In [16]:
df['alarm'].mode()

0    False
Name: alarm, dtype: object

In [17]:
df['alarm'] = df['alarm'].fillna(False).astype(bool)

In [18]:
print('Missing values in "alarm" column after filling:', df['alarm'].isnull().sum())

Missing values in "alarm" column after filling: 0


### "areaSqm":

No missing values, so no action is required.

### "balcony":

**1.1%** of the total values is missing. These values will be filled with the **most common category**.

In [19]:
df['balcony'].mode()

0    True
Name: balcony, dtype: object

In [20]:
df['balcony'] = df['balcony'].fillna(True).astype(bool)

In [21]:
print('Missing values in "balcony" column after filling:', df['balcony'].isnull().sum())

Missing values in "balcony" column after filling: 0


### "rooms":

**2.2%** of the total values is missing. The number of missing rows is very small, but because this column captures the number of rooms and has a bigger impact than others (e.g. airConditioning), we must decide a more intelligent way to fill the missing values.

In [22]:
df['rooms'].mode()

0    2.0
Name: rooms, dtype: float64

A 250sqm house with a missing room count is much more likely to have 4+ rooms than the global mode of 2.

In [23]:
# First check if median = mode
df['rooms'].median() == df['rooms'].mode()

0    True
Name: rooms, dtype: bool

By using pd.cut, we will create logical "bins" for the square footage, ensuring a 200sqm house doesn't get the same room count as a 40sqm studio.

In [24]:
bins = [0, 50, 100, 150, 200, 300, df['areaSqm'].max()]
labels = ['0-50', '50-100', '100-150', '150-200', '200-300', '300+']

df['area_bins'] = pd.cut(df['areaSqm'], bins=bins, labels=labels)

df['rooms'] = df['rooms'].fillna(df.groupby('area_bins', observed=False)['rooms'].transform('median'))

# If any are still NaN, we will just fill with global median and drop temp column
df['rooms'] = df['rooms'].fillna(df['rooms'].median())
df = df.drop(columns=['area_bins'])

In [25]:
print('Missing values in "rooms" column after smart filling:', df['rooms'].isnull().sum())

Missing values in "rooms" column after smart filling: 0


### "bathrooms":

**3%** of the total values is missing. Instead of filling every missing value with a global number like the column's mode, we will fill it based on the number of rooms.

In [26]:
df['bathrooms'].median() == df['bathrooms'].mode()

0    True
Name: bathrooms, dtype: bool

In [27]:
df['bathrooms'] = df['bathrooms'].fillna(df.groupby('rooms')['bathrooms'].transform('median'))

df['bathrooms'] = df['bathrooms'].fillna(df['bathrooms'].median())

In [28]:
print('Missing values in "bathrooms" column after filling:', df['bathrooms'].isnull().sum())

Missing values in "bathrooms" column after filling: 0


### "closedParkingSpots":

**Almost all the values are missing**. This column can't provide any useful information so it **must be dropped**.

In [29]:
df = df.drop(columns=['closedParkingSpots'])

### "currency":

The currency column only has 'EUR' values, indicating that price is counted in euros. We don't need this column in our dataframe so it **must be dropped**.

In [30]:
df = df.drop(columns=['currency'])

### "elevator":

**1%** of the total values is missing. These values will be filled with the **most common category**.

In [31]:
df['elevator'].mode()

0    True
Name: elevator, dtype: object

In [32]:
df['elevator'] = df['elevator'].fillna(True).astype(bool)

In [33]:
print('Missing values in "elevator" column after filling:', df['elevator'].isnull().sum())

Missing values in "elevator" column after filling: 0


### "energyClass":

**A very small amount** of the total values is missing. These values will be filled with the **most common category**.

In [34]:
df['energyClass'].value_counts()

energyClass
D                 2412
E                 1975
F                 1894
C                 1618
G                 1247
B                  885
A                  750
A+                 733
B+                 293
H                  255
Under issuance     143
Name: count, dtype: int64

In [35]:
df['energyClass'] = df['energyClass'].replace('Under issuance', np.nan)

In [36]:
df['energyClass'] = df['energyClass'].fillna('D')

In [37]:
print('Missing values in "energyClass" column after filling:', df['energyClass'].isnull().sum())

Missing values in "energyClass" column after filling: 0


### "fireplace":

**2.5%** of the total values is missing. These values will be filled with the **most common category**.

In [38]:
df['fireplace'].mode()

0    False
Name: fireplace, dtype: object

In [39]:
df['fireplace'] = df['fireplace'].fillna(False).astype(bool)

In [40]:
print('Missing values in "fireplace" column after filling:', df['fireplace'].isnull().sum())

Missing values in "fireplace" column after filling: 0


### "floor":

No missing values, so no action is required.

### "furnished":

**2.3%** of the total values is missing. These values will be filled with the **most common category**.

In [41]:
df['furnished'].value_counts()

furnished
no     9441
yes    2489
Name: count, dtype: int64

In [42]:
# Replace no --> False and yes --> True
df['furnished'] = df['furnished'].map({'yes': True, 'no': False})
df['furnished'].value_counts()

furnished
False    9441
True     2489
Name: count, dtype: int64

In [43]:
df['furnished'].mode()

0    False
Name: furnished, dtype: object

In [44]:
df['furnished'] = df['furnished'].fillna(False).astype(bool)

In [45]:
print('Missing values in "furnished" column after filling:', df['furnished'].isnull().sum())

Missing values in "furnished" column after filling: 0


### "garden":

**2.8%** of the total values is missing. These values will be filled with the **most common category**.

In [46]:
df['garden'].mode()

0    False
Name: garden, dtype: object

In [47]:
df['garden'] = df['garden'].fillna(False).astype(bool)

In [48]:
print('Missing values in "garden" column after filling:', df['garden'].isnull().sum())

Missing values in "garden" column after filling: 0


### "hasParking":

**2.5%** of the total values is missing. These values will be filled with the **most common category**.

In [49]:
df['hasParking'].mode()

0    False
Name: hasParking, dtype: object

In [50]:
df['hasParking'] = df['hasParking'].fillna(False).astype(bool)

In [51]:
print('Missing values in "hasParking" column after filling:', df['hasParking'].isnull().sum())

Missing values in "hasParking" column after filling: 0


### "isNewDevelopment":

No missing values, but we **don't need this column** for the analysis so it **must be dropped**.

In [52]:
df = df.drop(columns=['isNewDevelopment'])

### "kitchens":

**12%** of the total values is missing. Since the percentage is more than 10%, we have to examine this column further to decide the best strategy.

Instead of filling every missing value with a global number like the column's mode, we will fill it based on the number of rooms.

A missing value in a 1 room studio is almost certainly 1, while a missing value in a 6 room villa in Ekali might actually be 2.

In [53]:
df['kitchens'].median() == df['kitchens'].mode()

0    True
Name: kitchens, dtype: bool

In [54]:
df['kitchens'] = df['kitchens'].fillna(df.groupby('rooms')['kitchens'].transform('median'))

df['kitchens'] = df['kitchens'].fillna(df['kitchens'].median())

In [55]:
print('Missing values in "kitchens" column after filling:', df['kitchens'].isnull().sum())

Missing values in "kitchens" column after filling: 0


### "latitude":

No missing values, so no action is required.

### "livingRooms":

**25.7%** of the total values is missing; This is a substantial gap that demands a thoughtful approach. While the volume is high, it remains within a range where sophisticated imputation can still recover the column’s value without compromising the overall dataset.

We will follow the same approach used for the "kitchens" column.

In [56]:
df['livingRooms'].median() == df['livingRooms'].mode()

0    True
Name: livingRooms, dtype: bool

In [57]:
df['livingRooms'] = df['livingRooms'].fillna(df.groupby('rooms')['livingRooms'].transform('median'))

df['livingRooms'] = df['livingRooms'].fillna(df['livingRooms'].median())

In [58]:
print('Missing values in "livingRooms" column after filling:', df['livingRooms'].isnull().sum())

Missing values in "livingRooms" column after filling: 0


### "location":

No missing values, so no action is required.

### "longitude":

No missing values, so no action is required.

### "openParkingSpots":

**Almost all the values** are missing. This column cant provide any useful information so it **must be dropped**.

In [59]:
df = df.drop(columns=['openParkingSpots'])

### "pool":

**4.1%** of the total values is missing. These values will be filled with the **most common category**.

In [60]:
df['pool'].mode()

0    False
Name: pool, dtype: object

In [61]:
df['pool'] = df['pool'].fillna(False).astype(bool)

In [62]:
print('Missing values in "pool" column after filling:', df['pool'].isnull().sum())

Missing values in "pool" column after filling: 0


### "price":

No missing values, so no action is required.

### "pricePerSqm":

No missing value. But lets make sure about something in this column.

Since we web-scraped our data, the pricePerSqm might occasionally be calculated incorrectly in the source.

We will re-calculate this column to ensure absolute consistency.

In [63]:
df['pricePerSqm'] = df['price'] / df['areaSqm']

# Round to 2 decimal places
df['pricePerSqm'] = df['pricePerSqm'].round(2)

print('Null values in pricePerSqm after calculation:', df['pricePerSqm'].isnull().sum())

Null values in pricePerSqm after calculation: 0


### "yearBuilt":

**1.4%** of the total values is missing. Lets take a look at some of the values of this column.

For the missing values in the yearBuilt column, we chose not to use statistical 'fill-ins.' Since the age of a house is a deal-breaker in real estate pricing, we decided to drop these specific rows. We’d rather work with a slightly smaller, more accurate dataset than one filled with guesswork."

In [64]:
df = df.dropna(subset=['yearBuilt'])

df = df.reset_index(drop=True)

In [65]:
print('Number of missing values in "yearBuilt" column after filtering:', df['yearBuilt'].isnull().sum())

Number of missing values in "yearBuilt" column after filtering: 0


### "renovationYear":

**60%** of the total values is missing. Lets take a look at some of the values of this column.

In [66]:
df['renovationYear'].unique()

array([2020.,   nan, 2023., 2025., 2026., 2017., 2019., 2018., 2022.,
       2024., 2010., 2000., 1980., 2008., 2021., 2015., 2006., 2004.,
       2011., 1976., 1996., 2009., 2016., 2005., 2012., 1997., 1995.,
       1962., 2014., 2002., 1998., 2007., 1990., 1920., 1988., 1985.,
       1972., 1963., 1968., 1957., 1970., 1979., 1969., 2013., 2003.,
       1975., 1967., 1999., 2001., 1991., 1960., 1933., 1971., 1994.,
       1992., 1951., 1981.])

**The Reality Check: Is it "Missing" or "Never Happened"?**

If a house was renovated, the owner shouts it in the listing to justify a higher price. If it's blank, the house likely hasn't been renovated since it was built.

So missing values will not be treated as "Not Specified" or "Unknown". They will be treated as "**Not Renovated**".

We will create a **new column** that simply says "**Is this house renovated** ?" for houses renovated after 2018

In [67]:
df['is_renovated'] = df['renovationYear'] >= 2020

print(df['is_renovated'].value_counts())

is_renovated
False    8495
True     3554
Name: count, dtype: int64


We also will fill the missing values with the number "**0**".

In [68]:
df['renovationYear'] = df['renovationYear'].fillna(-1.0)

In [69]:
print('Number of missing values in "renovationYear" column after filtering:', df['renovationYear'].isnull().sum())

Number of missing values in "renovationYear" column after filtering: 0


### "solarHeater":

**4.5%** of the total values is missing. These values will be filled with the **most common category**.

In [70]:
df['solarHeater'].mode()

0    False
Name: solarHeater, dtype: object

In [71]:
df['solarHeater'] = df['solarHeater'].fillna(False).astype(bool)

In [72]:
print('Missing values in "solarHeater" column after filling:', df['solarHeater'].isnull().sum())

Missing values in "solarHeater" column after filling: 0


### "storage":

**2.2%** of the total values is missing. These values will be filled with the **most common category**.

In [73]:
df['storage'].mode()

0    False
Name: storage, dtype: object

In [74]:
df['storage'] = df['storage'].fillna(False).astype(bool)

In [75]:
print('Missing values in "storage" column after filling:', df['storage'].isnull().sum())

Missing values in "storage" column after filling: 0


### "view":

**5.8%** of the total values is missing. These values will be filled with the **most common category**.

In [76]:
df['view'].mode()

0    True
Name: view, dtype: object

In [77]:
df['view'] = df['view'].fillna(True).astype(bool)

In [78]:
print('Missing values in "view" column after filling:', df['view'].isnull().sum())

Missing values in "view" column after filling: 0


# Dataset Validation

In [79]:
# Data types of cleaned dataset
df.dtypes

airConditioning       bool
alarm                 bool
areaSqm              int64
balcony               bool
bathrooms          float64
elevator              bool
energyClass         object
fireplace             bool
floor                int64
furnished             bool
garden                bool
hasParking            bool
kitchens           float64
latitude           float64
livingRooms        float64
location            object
longitude          float64
pool                  bool
price                int64
pricePerSqm        float64
renovationYear     float64
rooms              float64
solarHeater           bool
storage               bool
view                  bool
yearBuilt          float64
is_renovated          bool
dtype: object

In [80]:
# Missing values of cleaned dataset
df.isnull().sum()

airConditioning    0
alarm              0
areaSqm            0
balcony            0
bathrooms          0
elevator           0
energyClass        0
fireplace          0
floor              0
furnished          0
garden             0
hasParking         0
kitchens           0
latitude           0
livingRooms        0
location           0
longitude          0
pool               0
price              0
pricePerSqm        0
renovationYear     0
rooms              0
solarHeater        0
storage            0
view               0
yearBuilt          0
is_renovated       0
dtype: int64

In [81]:
print('The shape of the cleaned dataset is:', df.shape)

The shape of the cleaned dataset is: (12049, 27)


# Dataset Exportation

In [82]:
path = r"C:\Users\marko\Documents\Database\Athens_House_Listings\Cleaned\house_listings_dataset_cleaned.csv"

In [83]:
# Save the cleaned dataset with UTF-8 encoding to ensure greek characters display correctly
df.to_csv(path, encoding='utf-8-sig', index=False)